# Metodología 2 — Embeddings (similitud semántica)

Cada chunk y cada frase "anchor" se convierten en vectores con el mismo modelo.
El puntaje de un chunk es qué tanto más cerca está de los anchors de
**incertidumbre** que de los de **certeza**.

Los anchors son parámetros: escribir otro conjunto mide otro concepto sin tocar
`src/`.

**Este es el notebook caro.** Embeber los ~4,600 sub-chunks toma varios minutos en
CPU, así que los vectores se guardan en caché: la segunda corrida los carga en
un segundo. **El caché viene incluido**
(`data/processed/banxico/*_chunk_embeddings.npy`), así que no hay que calcularlos.

La primera vez descarga el modelo `intfloat/multilingual-e5-base` (~1 GB) de Hugging Face.

Escribe `banxico_{anuncios,minutas}_index_embeddings.parquet` y `..._embedding_parent_scores.parquet`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# El código vive en src/, agrupado en corpus/ index/ analysis/ (sin __init__.py):
# hay que agregar cada subcarpeta a la ruta de Python. Ver src/paths.py.
sys.path.insert(0, str(Path.cwd().parent / "src"))
import paths
paths.add_src_subfolders_to_path()

from embeddings import (
    MPU_POSITIVE_ANCHORS,
    MPU_NEGATIVE_ANCHORS,
    embed_anchors,
    embed_chunks,
    score_chunks,
)
from common import aggregate_by_top_decile, standardize_to_index

MODEL_NAME = "intfloat/multilingual-e5-base"

In [2]:
# Los dos corpus. Cada uno produce su propio índice: un valor por documento.
CORPORA = ["banxico_anuncios", "banxico_minutas"]

## Los anchors

In [3]:
print("INCERTIDUMBRE (positivos):")
for anchor in MPU_POSITIVE_ANCHORS:
    print(f"  - {anchor}")
print("\nCERTEZA (negativos):")
for anchor in MPU_NEGATIVE_ANCHORS:
    print(f"  - {anchor}")

INCERTIDUMBRE (positivos):
  - Existe incertidumbre sobre la trayectoria futura de la inflación y el momento en que convergerá a la meta.
  - Las expectativas de inflación podrían desanclarse, lo que complica la conducción de la política monetaria.
  - No hay certeza sobre la trayectoria futura de la tasa de referencia ni sobre el momento de los próximos ajustes.
  - El balance de riesgos para la inflación está sesgado al alza y su evolución es difícil de anticipar.
  - La evolución de la actividad económica es incierta y podría alterar la postura de política monetaria.
  - La volatilidad del tipo de cambio representa un riesgo para la inflación y para las decisiones de política monetaria.
  - Choques externos y condiciones financieras globales más restrictivas podrían modificar el rumbo de la política monetaria.

CERTEZA (negativos):
  - La Junta de Gobierno decidió mantener la tasa de referencia, en línea con lo previsto y sin cambios en el panorama.
  - La inflación evolucionó confo

## Embeber los chunks

Los vectores dependen **solo del texto y del modelo**, no de los anchors. Por eso
se pueden cachear y reutilizar aunque se cambien los anchors: solo hay que
re-embeber los anchors, que son unas cuantas frases.

In [4]:
embedding_model = SentenceTransformer(MODEL_NAME)
positive = embed_anchors(embedding_model, MPU_POSITIVE_ANCHORS)
negative = embed_anchors(embedding_model, MPU_NEGATIVE_ANCHORS)

scored = {}
for corpus in CORPORA:
    chunks = pd.read_parquet(paths.PROCESSED_DIR / f"{corpus}_chunks.parquet")
    chunks = chunks.sort_values(["publication_date", "document_id", "chunk_number"]).reset_index(drop=True)

    cache = paths.PROCESSED_DIR / f"{corpus}_chunk_embeddings.npy"
    if cache.exists():
        chunk_embeddings = np.load(cache)
        assert len(chunk_embeddings) == len(chunks), "el caché no coincide con los chunks; borrar el .npy y re-correr"
        print(f"{corpus}: caché, {len(chunk_embeddings)} vectores cargados")
    else:
        print(f"{corpus}: calculando embeddings (varios minutos)...")
        chunk_embeddings = embed_chunks(embedding_model, chunks["chunk_text"])
        np.save(cache, chunk_embeddings)

    scored[corpus] = score_chunks(chunks, chunk_embeddings, positive, negative).rename(columns={
        "semantic_uncertainty_score": "mpu_score",
        "positive_similarity": "mpu_positive_similarity",
        "negative_similarity": "mpu_negative_similarity",
    })
scored["banxico_anuncios"][["chunk_id", "mpu_positive_similarity", "mpu_negative_similarity", "mpu_score"]].head()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

banxico_anuncios: caché, 960 vectores cargados
banxico_minutas: caché, 4003 vectores cargados


,chunk_id,mpu_positive_similarity,mpu_negative_similarity,mpu_score
0,banxico_apm_2000_01_18_c0001_s01,0.848656,0.854721,-0.006065
1,banxico_apm_2000_01_18_c0001_s02,0.826977,0.831676,-0.004700
2,banxico_apm_2000_05_16_c0001_s01,0.832592,0.835744,-0.003152
3,banxico_apm_2000_05_16_c0001_s02,0.849446,0.844267,0.005179
4,banxico_apm_2000_05_16_c0002_s01,0.828214,0.824703,0.003512


## Revisión manual

Antes de agregar nada, vale la pena leer los chunks en los extremos: si los más
altos no suenan inciertos, el problema está en los anchors, no en la agregación.

In [5]:
for corpus, scored_chunks in scored.items():
    for titulo, rows in [("MÁS ALTOS", scored_chunks.nlargest(2, "mpu_score")),
                         ("MÁS BAJOS", scored_chunks.nsmallest(2, "mpu_score"))]:
        for row in rows.itertuples(index=False):
            print("=" * 80)
            print(f"{corpus} · {titulo} · {row.chunk_id}   score {row.mpu_score:+.4f}")
            print(row.chunk_text[:600])
            print()

banxico_anuncios · MÁS ALTOS · banxico_apm_2018_06_21_c0001_s02   score +0.0651
en la volatilidad. Dicho comportamiento estuvo influenciado tanto por el entorno externo adverso antes descrito, como por otros factores, tales como la falta de acuerdos en el proceso de renegociación del TLCAN y la incertidumbre asociada al proceso electoral. Por su parte, las tasas de interés presentaron incrementos, especialmente las de menores plazos.

banxico_anuncios · MÁS ALTOS · banxico_apm_2018_11_15_c0003_s02   score +0.0473
otro lado, la inflación también se podría ver afectada en caso de que persistan presiones en los precios de los energéticos o incrementos en los de los productos agropecuarios; que se presente un escalamiento de medidas proteccionistas y compensatorias a nivel global; y un deterioro de las finanzas públicas. Además, por los choques observados y los niveles que ha alcanzado la inflación, existe el riesgo de que se presenten efectos de segundo orden en la formación de precios. T

## Agregación: de chunks a un número por documento

En anuncios y minutas el índice del documento es el **promedio simple** de sus chunks: al construir
esos corpus ya se quitó lo que no es discusión del panorama (tabla de pronósticos, anexo del staff,
lista de asistentes), y con ~2–4 chunks por anuncio el decil superior sería un solo chunk.
El promedio del **decil superior** se calcula también y queda a la vista (`mpu_embed_top_decile`).

Cada documento es un punto de su propia serie: no se agrega a trimestres aquí.

In [6]:
for corpus, scored_chunks in scored.items():
    # aggregate_by_top_decile calcula las dos cosas: el decil superior y el promedio simple
    embedding_index = aggregate_by_top_decile(scored_chunks, score_column="mpu_score", output_column="mpu_embed")
    # el promedio simple es la serie principal; el decil superior se conserva con su propio nombre
    embedding_index = embedding_index.rename(columns={
        "mpu_embed": "mpu_embed_top_decile", "mpu_embed_plain_mean": "mpu_embed"})
    embedding_index["idx_mpu_embed_top_decile"] = standardize_to_index(embedding_index["mpu_embed_top_decile"])
    embedding_index["idx_mpu_embed"] = standardize_to_index(embedding_index["mpu_embed"])
    print(f"{corpus}: correlación entre promedio simple y decil superior: "
          f"{embedding_index['mpu_embed'].corr(embedding_index['mpu_embed_top_decile']):.3f}")

    (embedding_index.sort_values("publication_date").reset_index(drop=True)
     .to_parquet(paths.PROCESSED_DIR / f"{corpus}_index_embeddings.parquet", index=False))

    # puntaje por chunk padre (la unidad que califica el LLM)
    parent_scores = scored_chunks.groupby("parent_chunk_id", as_index=False).agg(
        mpu_score=("mpu_score", "mean"), n_subchunks=("mpu_score", "size"))
    parent_scores.to_parquet(paths.PROCESSED_DIR / f"{corpus}_embedding_parent_scores.parquet", index=False)
    print(f"  guardado: {len(embedding_index)} documentos, {len(parent_scores)} chunks padre")

banxico_anuncios: correlación entre promedio simple y decil superior: 0.703
  guardado: 238 documentos, 578 chunks padre
banxico_minutas: correlación entre promedio simple y decil superior: 0.590
  guardado: 126 documentos, 2113 chunks padre
